In [1]:
%pip install --no-warn-conflicts -q \
  "docling" \
  "transformers==4.49.0" \
  "accelerate" \
  "torch" \
  "torchvision" \
  "huggingface_hub>=0.23,<1" \
  "pillow"

%pip install --no-warn-conflicts -q paddlepaddle
%pip install --no-warn-conflicts -q "paddleocr[all]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.7/266.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━

In [2]:
%pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 61.4 MB/s eta 0:00:00


In [3]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

user_context = """
Estoy estudiando ciencia de datos y esta asignatura me cuesta.
Quiero una explicación clara, paso a paso, con ejemplos sencillos.
"""

Saving 3 diapos.pdf to 3 diapos.pdf


In [4]:
from google.colab import files

uploaded_audio = files.upload()
audio_path = list(uploaded_audio.keys())[0]

print("Audio subido:", audio_path)

Saving audio 3 diapos.m4a to audio 3 diapos.m4a
Audio subido: audio 3 diapos.m4a


# Convertimos PDF a Texto con OCR de Docling

In [5]:
from docling.document_converter import DocumentConverter
import json

def extract_with_docling(file_path: str) -> str:
    converter = DocumentConverter()
    result = converter.convert(file_path)
    return result.document.export_to_markdown()

#Convierte a JSON el formato y cada pagina es cogida como Rmarkdown
#Formato tipo :{"page":1}
def extract_with_docling_2(file_path: str):
    converter = DocumentConverter()
    result = converter.convert(file_path)

    pages_json = []
    for i, page in enumerate(result.document.pages, start=1):
        #page_markdown = page.export_to_markdown()
        page_markdown = result.document.export_to_markdown(page_no=i)
        pages_json.append({
            "pag": i,
            "content": page_markdown,
            "audio_segment": "",
            "resumenIA": ""
        })

    return pages_json

#pdf_text = extract_with_docling(pdf_path)
pdf_text = extract_with_docling_2(pdf_path)

json_path = "texto_slides.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(pdf_text, f, ensure_ascii=False, indent=4)

print(pdf_text[:2000])

[INFO] 2026-03-24 13:25:31,178 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-24 13:25:31,196 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-24 13:25:31,197 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-24 13:25:31,307 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-24 13:25:31,310 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-24 13:25:31,311 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-24 13:25:31,359 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-24 13:25:31,390 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

[{'pag': 1, 'content': '## Linear models: recursive least squares (RLS)\n\nStandard least squares: f ( x n ) = x ⊤ n w , loss function ( f ( x n ) -y n ) 2 , covariance matrix Σ n = X ⊤ X , X ∈ R n × d . The solution for f ∗ ( x ) = x ⊤ w ∗ is\n\n<!-- formula-not-decoded -->\n\n.\n\nRLS: based on the matrix inversion lemma: Initialize w 0 = 0 ∈ R d , and Γ 0 = I ∈ R d × d Iteration:\n\n<!-- formula-not-decoded -->\n\nIt can be shown that Γ n = Σ -1 n .\n\nThe regularized version is obtained as usual by introducing a regularizer in the loss function, ∑ i = 1 n ( w ⊤ x i -y i ) 2 + λ ∥ w ∥ 2 2 , and starting from Γ 0 = ( I + λ ) -1 . This converges to Γ n = (Σ n + λ I ) -1 .\n\n<!-- image -->\n\n<!-- formula-not-decoded -->\n\n<!-- image -->', 'audio_segment': '', 'resumenIA': ''}, {'pag': 2, 'content': '## Stochastic Gradient Descent\n\nInstead of updating w n using\n\n<!-- formula-not-decoded -->\n\nuse gradient descent:\n\n<!-- image -->\n\n<!-- formula-not-decoded -->\n\nwhere α n is

# Convertimos AUDIO a Texto con WhisperModel

In [6]:
from faster_whisper import WhisperModel
import json

audio_model_size = "small"

audio_model = WhisperModel(
    audio_model_size,
    device="cuda",
    compute_type="float16"
)

segments, info = audio_model.transcribe(
    audio_path,
    language="es",
    beam_size=5
)

print("Idioma detectado:", info.language)
print("Probabilidad idioma:", info.language_probability)

audio_segments = []
audio_text_parts = []

for segment in segments:
    audio_segments.append({
        "start": segment.start,
        "end": segment.end,
        "text": segment.text
    })
    audio_text_parts.append(segment.text)

    print(f"[{segment.start:.2f}s - {segment.end:.2f}s] {segment.text}")

audio_text = "\n".join(audio_text_parts)

with open("audio_transcripcion.txt", "w", encoding="utf-8") as f:
    f.write(audio_text)

with open("audio_segmentos.json", "w", encoding="utf-8") as f:
    json.dump(audio_segments, f, ensure_ascii=False, indent=2)

print("Guardado en audio_transcripcion.txt")
print("Guardado en audio_segmentos.json")

Idioma detectado: es
Probabilidad idioma: 1
[0.00s - 5.32s]  más de detalle. Primero, la idea del recursive least squares RLS es que tenemos un modelo
[5.32s - 10.32s]  lineal que predice a partir de una combinación de las variables. Lo que hace RLS es que cada
[10.32s - 14.64s]  vez que llega un nuevo dato, actualiza los parámetros del modelo sin volver a recalcular
[14.64s - 19.26s]  todo. Esto lo hace a través de fórmulas matriciales y hay una versión regularizada
[19.26s - 24.36s]  que evita que el modelo se sobreajuste. En definitiva, RLS es ideal para actualizar
[24.36s - 29.68s]  modelos lineales en tiempo real. En segundo lugar, aparece el stochastic gradient descent
[29.68s - 36.12s]  SGD. En vez de hacer actualizaciones matriciales, SGD ajusta los pesos poco a poco en la dirección
[36.12s - 42.24s]  que reduce el error usando una tasa de aprendizaje. Si la tasa es grande, puede volver sin estable,
[42.24s - 47.40s]  si es pequeña, más lento. Lo bueno es que no está limitado a

# Unimos texto de diapositiva con texto de audios con un LLM simple

In [1]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# -----------------------------
# 1️⃣ Configuración del modelo
# -----------------------------
#model_name = "HuggingFaceTB/SmolLM3-3B"  # SmolLM3-3B en Hugging Face
model_name_union = "Qwen/Qwen2.5-1.5B-Instruct"
#model_name_union = "mistralai/Mistral-7B-Instruct-v0.2"
#model_name_union = "microsoft/phi-2"
print("Cargando tokenizer y modelo...")
tokenizer = AutoTokenizer.from_pretrained(model_name_union)

model_union = AutoModelForCausalLM.from_pretrained(
    model_name_union,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("Modelo cargado ✅")

Cargando tokenizer y modelo...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo cargado ✅


In [4]:
def build_slide_discrimination_prompt(slides):
    slides_block = []
    for s in slides:
        slides_block.append(f"Página {s['pag']}:\n{s['content']}\n")
    slides_text = "\n".join(slides_block)

    return f"""
Analiza el conjunto completo de diapositivas y discrimina claramente los temas de cada una.

Diapositivas:
{slides_text}

Devuelve solo JSON válido con este formato:
[
  {{
    "pag": 1,
    "topic": "...",
    "keywords": ["...", "..."],
    "exclusive_keywords": ["...", "..."],
    "summary": "..."
  }}
]

Reglas:
- Usa un objeto por diapositiva.
- "exclusive_keywords" debe contener términos que ayuden a distinguir esa diapositiva de las otras.
- "summary" debe ser breve y centrado en el tema principal.
"""



In [5]:
import re
import json
import torch

# -----------------------------
# 2️⃣ Función para limpiar slides
# -----------------------------
def clean_slide(text):
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# -----------------------------
# 3️⃣ Obtener perfiles temáticos de TODAS las slides
# -----------------------------
def get_slide_profiles(slides, tokenizer, model):
    cleaned_slides = []
    for s in slides:
        cleaned_slides.append({
            "pag": s["pag"],
            "content": clean_slide(s["content"])
        })

    prompt = build_slide_discrimination_prompt(cleaned_slides)

    messages = [
        {"role": "system", "content": "Responde solo con JSON válido."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print("RESPUESTA RAW PERFILES:")
    print(result)
    print("-" * 80)

    try:
        profiles = json.loads(result)
        if isinstance(profiles, list):
            return profiles
    except Exception:
        pass

    match = re.search(r"\[\s*{.*}\s*\]", result, flags=re.DOTALL)
    if match:
        try:
            profiles = json.loads(match.group(0))
            if isinstance(profiles, list):
                return profiles
        except Exception:
            pass

    raise ValueError("No se pudo parsear el JSON de perfiles de slides.")


# -----------------------------
# 4️⃣ Prompt para clasificar un segmento de audio
# -----------------------------
def build_audio_classification_prompt(segment, slide_profiles):
    profiles_block = []
    for p in slide_profiles:
        profiles_block.append(
            f"""Página {p['pag']}:
Tema: {p.get('topic', '')}
Keywords: {", ".join(p.get('keywords', []))}
Exclusive keywords: {", ".join(p.get('exclusive_keywords', []))}
Resumen: {p.get('summary', '')}
"""
        )

    profiles_text = "\n".join(profiles_block)

    return f"""
Eres un asistente que asigna segmentos de audio a la diapositiva más adecuada.

Perfiles de diapositivas:
{profiles_text}

Segmento de audio:
[{segment['start']:.2f}-{segment['end']:.2f}] {segment['text']}

Tarea:
Decide a qué página pertenece mejor este segmento de audio.

Devuelve solo JSON válido con este formato:
{{
  "assigned_page": 1,
  "confidence": 0.95
}}

Reglas:
- "assigned_page" debe ser el número de página más adecuado.
- Si el segmento no encaja claramente, devuelve:
  {{
    "assigned_page": null,
    "confidence": 0.0
  }}
- No escribas nada fuera del JSON.
"""


# -----------------------------
# 5️⃣ Clasificar cada segmento de audio
# -----------------------------
def classify_audio_segment(segment, slide_profiles, tokenizer, model):
    prompt = build_audio_classification_prompt(segment, slide_profiles)

    messages = [
        {"role": "system", "content": "Responde solo con JSON válido."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    try:
        parsed = json.loads(result)
        return parsed
    except Exception:
        match = re.search(r"\{.*\}", result, flags=re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                return parsed
            except Exception:
                pass

    return {"assigned_page": None, "confidence": 0.0}


# -----------------------------
# 6️⃣ Procesamiento
# -----------------------------
with open("texto_slides.json", "r", encoding="utf-8") as f:
    slides_json = json.load(f)

with open("audio_segmentos.json", "r", encoding="utf-8") as f:
    audio_segments = json.load(f)

# 6.1 Obtener perfiles globales de las slides
slide_profiles = get_slide_profiles(slides_json, tokenizer, model_union)

print("PERFILES DE SLIDES:")
print(json.dumps(slide_profiles, ensure_ascii=False, indent=2))
print("=" * 100)

# 6.2 Inicializar estructura por página
slides_aligned_dict = {}

for slide in slides_json:
    page_num = slide["pag"]
    slide_text = clean_slide(slide["content"])

    slides_aligned_dict[page_num] = {
        "pag": page_num,
        "slide_text": slide_text,
        "profile": next((p for p in slide_profiles if p["pag"] == page_num), {}),
        "selected_indices": [],
        "audio_segments": [],
        "audio_text": ""
    }

# 6.3 Clasificar cada segmento de audio
for idx, seg in enumerate(audio_segments):
    classification = classify_audio_segment(seg, slide_profiles, tokenizer, model_union)

    assigned_page = classification.get("assigned_page", None)
    confidence = classification.get("confidence", 0.0)

    print(f"Segmento {idx} -> página {assigned_page} (conf={confidence})")

    if assigned_page is not None and assigned_page in slides_aligned_dict:
        slides_aligned_dict[assigned_page]["selected_indices"].append(idx)
        slides_aligned_dict[assigned_page]["audio_segments"].append(seg)

# 6.4 Construir texto final por slide
slides_aligned = []
for page_num in sorted(slides_aligned_dict.keys()):
    item = slides_aligned_dict[page_num]
    item["audio_text"] = " ".join(seg["text"] for seg in item["audio_segments"]).strip()
    slides_aligned.append(item)

# 6.5 Guardar salida principal
with open("slides_aligned_semantic.json", "w", encoding="utf-8") as f:
    json.dump(slides_aligned, f, ensure_ascii=False, indent=2)

print("✅ Guardado en slides_aligned_semantic.json")

# 6.6 Actualizar slides_json si quieres mantener el otro formato
for slide in slides_json:
    page_num = slide["pag"]
    aligned = slides_aligned_dict.get(page_num, {})
    slide["audio_segment"] = aligned.get("audio_text", "")
    slide["resumenIA"] = ""
    slide["selected_indices"] = aligned.get("selected_indices", [])

with open("slides_with_professor.json", "w", encoding="utf-8") as f:
    json.dump(slides_json, f, ensure_ascii=False, indent=2)

print("✅ JSON actualizado guardado en slides_with_professor.json")

RESPUESTA RAW PERFILES:
```json
[
  {
    "pag": 1,
    "topic": "Linear Models: Recursive Least Squares (RLS)",
    "keywords": ["Recursive Least Squares", "Least Squares", "Covariance Matrix", "Solution for f*"],
    "exclusive_keywords": ["Matrix Inversion Lemma", "Regularized Version"],
    "summary": "Describes RLS algorithm for linear models, including initialization, iteration process, and regularization."
  },
  {
    "pag": 2,
    "topic": "Stochastic Gradient Descent",
    "keywords": ["Gradient Descent", "Learning Rate", "Convex Loss Function", "Batches", "Order of Input Samples"],
    "exclusive_keywords": ["Batch Learning", "Deterministic Order"],
    "summary": "Explains how SGD updates weights using gradients, with options for batch processing and sample ordering."
  },
  {
    "pag": 3,
    "topic": "Kernel Methods: KRR (LS-SVM)",
    "keywords": ["Kernel Regression", "Weights Recursively Obtained", "Naive Kernel Regression"],
    "exclusive_keywords": ["Kernels", "Pred

In [6]:
import json

# -----------------------------
# 7️⃣ Construir salida limpia para otro LLM
# -----------------------------
def build_llm_ready_output(slides_aligned):
    llm_ready = {
        "document_context": {
            "num_slides": len(slides_aligned),
            "language": "es",
            "source": "pdf + transcripción de audio"
        },
        "slides": []
    }

    for slide in slides_aligned:
        profile = slide.get("profile", {})

        item = {
            "pag": slide.get("pag"),
            "topic": profile.get("topic", ""),
            "keywords": profile.get("keywords", []),
            "exclusive_keywords": profile.get("exclusive_keywords", []),
            "slide_text": slide.get("slide_text", ""),
            "audio_text": slide.get("audio_text", "").strip(),
            "combined_context": (
                f"TEMA: {profile.get('topic', '')}\n"
                f"PALABRAS CLAVE: {', '.join(profile.get('keywords', []))}\n"
                f"PALABRAS DISTINTIVAS: {', '.join(profile.get('exclusive_keywords', []))}\n"
                f"RESUMEN TEMÁTICO: {profile.get('summary', '')}\n\n"
                f"TEXTO DE LA DIAPOSITIVA:\n{slide.get('slide_text', '')}\n\n"
                f"EXPLICACIÓN DEL AUDIO:\n{slide.get('audio_text', '').strip()}"
            )
        }

        llm_ready["slides"].append(item)

    return llm_ready


# Construir JSON final limpio
llm_ready_output = build_llm_ready_output(slides_aligned)

# Guardar archivo para el siguiente LLM
with open("slides_for_next_llm.json", "w", encoding="utf-8") as f:
    json.dump(llm_ready_output, f, ensure_ascii=False, indent=2)

print("✅ Guardado en slides_for_next_llm.json")

# Vista rápida
print(json.dumps(llm_ready_output, ensure_ascii=False, indent=2)[:4000])

✅ Guardado en slides_for_next_llm.json
{
  "document_context": {
    "num_slides": 3,
    "language": "es",
    "source": "pdf + transcripción de audio"
  },
  "slides": [
    {
      "pag": 1,
      "topic": "Linear Models: Recursive Least Squares (RLS)",
      "keywords": [
        "Recursive Least Squares",
        "Least Squares",
        "Covariance Matrix",
        "Solution for f*"
      ],
      "exclusive_keywords": [
        "Matrix Inversion Lemma",
        "Regularized Version"
      ],
      "slide_text": "## Linear models: recursive least squares (RLS) Standard least squares: f ( x n ) = x ⊤ n w , loss function ( f ( x n ) -y n ) 2 , covariance matrix Σ n = X ⊤ X , X ∈ R n × d . The solution for f ∗ ( x ) = x ⊤ w ∗ is . RLS: based on the matrix inversion lemma: Initialize w 0 = 0 ∈ R d , and Γ 0 = I ∈ R d × d Iteration: It can be shown that Γ n = Σ -1 n . The regularized version is obtained as usual by introducing a regularizer in the loss function, ∑ i = 1 n ( w ⊤ x i -y

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 36.9 MB/s eta 0:00:00


In [9]:
import fitz
from paddleocr import PaddleOCR

def pdf_page_to_image(pdf_path: str, page_num: int = 0, zoom: float = 2.0) -> str:
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    out = f"page_{page_num+1}.png"
    pix.save(out)
    return out

def extract_with_paddleocr_from_first_page(pdf_path: str) -> str:
    img_path = pdf_page_to_image(pdf_path, 0)
    ocr = PaddleOCR()
    result = ocr.predict(img_path)
    return str(result)

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Modelo cargado")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo cargado


In [14]:
import json
import torch

# -----------------------------
# 9️⃣ Prompt por diapositiva
# -----------------------------
def build_single_slide_explanation_prompt(slide):
    return f"""
Eres un asistente académico. Explica esta diapositiva de forma clara, completa y útil para estudiar.

Instrucciones:
- Responde en español.
- Explica solo esta diapositiva.
- Usa tanto el texto de la diapositiva como el audio asociado.
- No inventes información.
- Si el audio complementa la diapositiva, intégralo en la explicación.
- Mantén una estructura clara.
- Termina la explicación completamente.

Datos de la diapositiva:

Número de diapositiva: {slide['pag']}
Tema: {slide['topic']}
Keywords: {", ".join(slide['keywords'])}
Keywords distintivas: {", ".join(slide['exclusive_keywords'])}

Texto de la diapositiva:
{slide['slide_text']}

Texto del audio asociado:
{slide['audio_text']}

Devuelve la respuesta con este formato:

### DIAPOSITIVA {slide['pag']}

**Tema:** ...

**Explicación:**
...
"""


next_llm_prompt = build_summary_prompt_for_next_llm(llm_ready_output)
'''
with open("prompt_for_next_llm.txt", "w", encoding="utf-8") as f:
    f.write(next_llm_prompt)

print("✅ Guardado en prompt_for_next_llm.txt")
print(next_llm_prompt[:3000])
'''

'\nwith open("prompt_for_next_llm.txt", "w", encoding="utf-8") as f:\n    f.write(next_llm_prompt)\n\nprint("✅ Guardado en prompt_for_next_llm.txt")\nprint(next_llm_prompt[:3000])\n'

In [15]:
prompt = next_llm_prompt

print(prompt[:3000])


Eres un asistente académico. Vas a explicar una presentación diapositiva a diapositiva.

Instrucciones:
- Explica detalladamente cada diapositiva por separado.
- Utiliza la información del texto de la diapositiva y del audio asociado para guiarte en la explicación.
- No inventes información.
- Explica de forma clara y útil para estudiar.
- Responde en español.
- Devuelve una salida estructurada por diapositiva.

Contenido:

DIAPOSITIVA 1
Tema: Linear Models: Recursive Least Squares (RLS)
Keywords: Recursive Least Squares, Least Squares, Covariance Matrix, Solution for f*
Keywords distintivas: Matrix Inversion Lemma, Regularized Version

Texto de la diapositiva:
## Linear models: recursive least squares (RLS) Standard least squares: f ( x n ) = x ⊤ n w , loss function ( f ( x n ) -y n ) 2 , covariance matrix Σ n = X ⊤ X , X ∈ R n × d . The solution for f ∗ ( x ) = x ⊤ w ∗ is . RLS: based on the matrix inversion lemma: Initialize w 0 = 0 ∈ R d , and Γ 0 = I ∈ R d × d Iteration: It can b

In [16]:
all_explanations = []

for slide in llm_ready_output["slides"]:
    prompt = build_single_slide_explanation_prompt(slide)

    messages = [
        {"role": "system", "content": "Responde siempre en español, de forma clara y académica."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    explanation = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print(f"\n{'='*80}")
    print(f"DIAPOSITIVA {slide['pag']}")
    print(f"{'='*80}")
    print(explanation)

    all_explanations.append(explanation)


# -----------------------------
# 1️⃣1️⃣ Unir y guardar
# -----------------------------
final_output = "\n\n".join(all_explanations)

with open("explicacion_pdf_audio.txt", "w", encoding="utf-8") as f:
    f.write(final_output)

print("\n✅ Guardado en explicacion_pdf_audio.txt")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



DIAPOSITIVA 1
### DIAPOSITIVA 1

**Tema:** Linear models: recursive least squares (RLS)

**Explicación:**

En el contexto de modelos lineales, el método de **Least Squares (LS)** proporciona una solución para minimizar la suma de los cuadrados de las diferencias entre las predicciones y los valores reales. Para un modelo lineal dado por \( f(x_n) = x_n^T w \), donde \( x_n \) es una muestra de entrada y \( w \) son los pesos desconocidos, la función de pérdida es \( (f(x_n) - y_n)^2 \). La matriz de covarianza \( \Sigma_n \) se define como \( \Sigma_n = X^T X \), donde \( X \) es una matriz de diseño de tamaño \( n \times d \).

El objetivo del **Recursive Least Squares (RLS)** es actualizar los pesos \( w \) de manera eficiente cuando se recibe un nuevo dato \( x_n \) y el valor correspondiente \( y_n \). El proceso inicializa los pesos \( w_0 \) como cero y la matriz de varianza \( \Gamma_0 \) como la identidad \( I \). A medida que se recogen más datos, se actualiza la matriz de va

In [18]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.5 MB/s eta 0:00:00


In [20]:
import re
import textwrap
import fitz  # PyMuPDF
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY
from reportlab.lib.units import cm
from reportlab.lib import colors

# =========================================================
# 1) Leer y separar explicaciones por diapositiva
# =========================================================
def parse_explanations(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    # separa por encabezados tipo: ### DIAPOSITIVA 1
    pattern = r"###\s+DIAPOSITIVA\s+(\d+)\s*"
    parts = re.split(pattern, text)

    explanations = {}
    # re.split devuelve: [texto_antes, num1, contenido1, num2, contenido2, ...]
    for i in range(1, len(parts), 2):
        slide_num = int(parts[i])
        content = parts[i + 1].strip()
        explanations[slide_num] = content

    return explanations


# =========================================================
# 2) Crear PDF de explicaciones
# =========================================================
def create_explanations_pdf(explanations, output_pdf):
    doc = SimpleDocTemplate(
        output_pdf,
        pagesize=A4,
        rightMargin=2*cm,
        leftMargin=2*cm,
        topMargin=2*cm,
        bottomMargin=2*cm
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        name="TitleCustom",
        parent=styles["Heading1"],
        fontSize=18,
        leading=22,
        textColor=colors.HexColor("#1f3b73"),
        spaceAfter=12
    )

    body_style = ParagraphStyle(
        name="BodyCustom",
        parent=styles["BodyText"],
        fontSize=11,
        leading=16,
        alignment=TA_JUSTIFY,
        spaceAfter=8
    )

    bold_style = ParagraphStyle(
        name="BoldCustom",
        parent=body_style,
        fontSize=11,
        leading=16,
        textColor=colors.black,
        spaceAfter=8
    )

    story = []

    for slide_num in sorted(explanations.keys()):
        raw = explanations[slide_num]

        # Convierte saltos dobles en párrafos
        blocks = [b.strip() for b in raw.split("\n\n") if b.strip()]

        story.append(Paragraph(f"Explicación — Diapositiva {slide_num}", title_style))
        story.append(Spacer(1, 0.3*cm))

        for block in blocks:
            # negritas simples tipo **Tema:**
            block = re.sub(r"\*\*(.*?)\*\*", r"<b>\1</b>", block)
            block = block.replace("\n", "<br/>")
            story.append(Paragraph(block, body_style))
            story.append(Spacer(1, 0.15*cm))

        story.append(PageBreak())

    if story and isinstance(story[-1], PageBreak):
        story.pop()

    doc.build(story)


# =========================================================
# 3) Intercalar PDF original + PDF de explicaciones
# =========================================================
def interleave_pdfs(original_pdf, explanations_pdf, output_pdf):
    original = fitz.open(original_pdf)
    expl = fitz.open(explanations_pdf)
    final = fitz.open()

    n = min(len(original), len(expl))

    for i in range(n):
        final.insert_pdf(original, from_page=i, to_page=i)
        final.insert_pdf(expl, from_page=i, to_page=i)

    # si hubiera más páginas en el original, las añade al final
    if len(original) > n:
        final.insert_pdf(original, from_page=n, to_page=len(original)-1)

    final.save(output_pdf)
    final.close()
    original.close()
    expl.close()


# =========================================================
# 4) Ejecutar
# =========================================================
txt_path = "explicacion_pdf_audio.txt"
original_pdf = "3 diapos.pdf"   # usa el PDF que subiste al principio
explanations_pdf = "explicaciones_slides.pdf"
final_pdf = "slides_y_explicacion_intercalado.pdf"

explanations = parse_explanations(txt_path)
create_explanations_pdf(explanations, explanations_pdf)
interleave_pdfs(original_pdf, explanations_pdf, final_pdf)

print(f"✅ PDF de explicaciones: {explanations_pdf}")
print(f"✅ PDF final intercalado: {final_pdf}")

✅ PDF de explicaciones: explicaciones_slides.pdf
✅ PDF final intercalado: slides_y_explicacion_intercalado.pdf
